In [1]:
import torch
from torch.utils.data import DataLoader
import torch.nn.functional as F

from rvsablation.utils import seed_everything, read_config
rng = seed_everything(42)

from rvsablation.utils import PrecompLatentTransitionDataset

from matrepr import mprint

TUBELETS = 4
BSIZE = 500

🌱 Global seed set to 42


# Context shift probe

Given two consecutive clips of tubelet groups $\left[g_1, g_2, g_3, g_4\right]$ and $\left[g_2, g_3, g_4, g_5\right]$ we can compute how the representation of the same group changes based on its position within the clip and thus its position in the causal context chain. 

## Load Transition Test Sets

In [2]:
dataset_ae = PrecompLatentTransitionDataset("../ae_runs/fsq_videovae_run_1/latents_transition/testing")
dataset_jepa = PrecompLatentTransitionDataset("../jepa_runs/fsq_v_jepa_run_1/latents_transition/testing")

loader_ae = DataLoader(dataset_ae, batch_size=BSIZE)
loader_jepa = DataLoader(dataset_jepa, batch_size=BSIZE)

(len(dataset_ae) == len(dataset_jepa)) * len(dataset_ae)

57500

In [3]:
errors = []

for b in loader_jepa:
    present = b['latents']
    future = b['future_latents']

    B, _, D = present.shape

    present = present.reshape(B, TUBELETS, -1, D)
    future = future.reshape(B, TUBELETS, -1, D)

    error = F.l1_loss(present[:, 1:], future[:, :-1], reduction="none").mean(dim=(-1, -2))
    errors.append(error)

errors = torch.cat(errors)
errors.mean(dim=0)

tensor([1.4280, 1.2456, 1.4136])

## Measure Context-Shift Drift & Latent Variance

In [4]:
def get_context_delta_i(dataloder):
    errors = []
    for b in dataloder:
        present = b['latents']
        future = b['future_latents']

        B, _, D = present.shape
        present = present.reshape(B, TUBELETS, -1, D)
        future = future.reshape(B, TUBELETS, -1, D)
        error = F.l1_loss(present[:, 1:], future[:, :-1], reduction="none").mean(dim=(-1, -2))
        errors.append(error)

    errors = torch.cat(errors)
    return errors.mean(dim=0), errors.std(dim=0) / errors.shape[0] ** 0.5


def get_position_delta_i(dataloader):
    errors = []
    for b in dataloader:
        present = b['latents']
        
        B, _, D = present.shape
        present = present.reshape(B, TUBELETS, -1, D)
        error = F.l1_loss(present[:, 1:], present[:, :-1], reduction="none").mean(dim=(-1, -2))
        errors.append(error)

    errors = torch.cat(errors)
    return errors.mean(dim=0), errors.std(dim=0) / errors.shape[0] ** 0.5

ae_sensitivity, ae_sensitivity_sem = get_context_delta_i(loader_ae)
jepa_sensitivity, jepa_sensitivity_sem = get_context_delta_i(loader_jepa)

sensitivity_matrix = torch.cat([ae_sensitivity[None, :], jepa_sensitivity[None, :]], dim=0)

mprint(sensitivity_matrix,
       col_labels=["z_2 - z_1'", "z_3 - z_2'", "z_4 - z_3'"],
       row_labels=["AE", "JEPA"],
       title=None)


ae_pos_sensitivity, ae_pos_sensitivity_sem = get_position_delta_i(loader_ae)
jepa_pos_sensitivity, jepa_pos_sensitivity_sem = get_position_delta_i(loader_jepa)

pos_sensitivity_matrix = torch.cat([ae_pos_sensitivity[None, :], jepa_pos_sensitivity[None, :]], dim=0)

mprint(pos_sensitivity_matrix,
       col_labels=["z_2 - z_1", "z_3 - z_2", "z_4 - z_3"],
       row_labels=["AE", "JEPA"],
       title=None)

sensitivity_ratio_matrix = sensitivity_matrix / pos_sensitivity_matrix
mprint(sensitivity_ratio_matrix,
       col_labels=["z_2", "z_3", "z_4"],
       row_labels=["AE", "JEPA"],
       title=None)

        z_2 - z_1'    z_3 - z_2'    z_4 - z_3'
     ┌                                          ┐
  AE │    0.1306       0.05041        0.0688    │
JEPA │    1.428         1.246         1.414     │
     └                                          ┘
        z_2 - z_1    z_3 - z_2    z_4 - z_3
     ┌                                       ┐
  AE │    1.177        1.166        1.174    │
JEPA │    1.572        1.487        1.583    │
     └                                       ┘
        z_2      z_3      z_4
     ┌                          ┐
  AE │ 0.1109  0.04322  0.05862 │
JEPA │ 0.9085  0.8376   0.8928  │
     └                          ┘


In [15]:
def get_latent_variance(dataloader):
    mean_variance = []
    for b in dataloader:
        present = b['latents']
        # print(present.var(dim=(0, 1)).shape)
        mean_variance.append(present.var(dim=(0, 1))[None, :])

    return torch.cat(mean_variance, dim=0)

ae_per_dim_var = get_latent_variance(loader_ae).mean(dim=0) 
jepa_per_dim_var = get_latent_variance(loader_jepa).mean(dim=0)

mprint(
    torch.cat([ae_per_dim_var[None, :], jepa_per_dim_var[None, :]], dim=0),
    row_labels=["AE", "JEPA"],
    col_labels=["Dim. 1", "Dim. 2", "Dim. 3", "Dim. 4", "Dim. 5", "Dim. 6"],
    title="Variance per dimension."
)

Variance per dimension.
        Dim. 1    Dim. 2    Dim. 3    Dim. 4    Dim. 5    Dim. 6
     ┌                                                            ┐
  AE │  8.254     7.391     6.865     2.122     2.075     2.111   │
JEPA │  6.216     6.483     6.381     1.902      1.83      1.91   │
     └                                                            ┘
